# Assignment 3: Self-Supervised and Transfer Learning

In this assignment, you will gain experience with PyTorch through learning how to use pretrained models provided by the deep learning community and adapt these models to new tasks and losses. You will pretrain a self-supervised rotation prediction model on the CIFAR10 dataset without using class labels, and then finetune this model for CIFAR10 classification.

![Self-Supervised Learning by Rotation Prediction on CIFAR10](https://slazebni.cs.illinois.edu/spring25/assignment3/rotation_prediction.png) <br>
*Source: [Gidaris et al. (2018)](https://arxiv.org/pdf/1803.07728.pdf)*

You will run and compare five main experiments:

1. Self-supervised rotation pretraining

2. Finetuning part of a pretrained model on CIFAR10 classification

3. Training a full model on CIFAR10 classification, initialized from random weights

4. Training a full model on CIFAR10 classification, initialized from pretrained weights

5. Experimenting with advanced models and techniques for CIFAR10 classification

You will implement the dataloaders as well as all training and finetuning steps in PyTorch based on the starter code.

Baselines:
| Part | Task | Initialization / Training Setup | Expected Test Accuracy |
|---|---|---|---|
| 1 | Rotation prediction (self-supervised) | ResNet18, train on rotation labels (0/90/180/270) | 78% |
| 2 | CIFAR10 classification (partial finetune) | Start from Part 1 checkpoint, train only `layer4` + `fc` | 60% |
| 3 | CIFAR10 classification (full model) | Random init, train all layers | 80% |
| 4 | CIFAR10 classification (full model) | Start from Part 1 checkpoint, train all layers | 82% |
| 5 | Advanced model on classification task | Better architecture + training tricks | >= +3% over your **best** model among Parts 2, 3, and 4 |

Optional self-study: [PyTorch Tutorial](https://docs.pytorch.org/tutorials/beginner/basics/intro.html).

## (Optional) Colab Setup
If you aren't using Colab, you can delete the following code cell. This is just to help students with mounting to Google Drive to access the other .py files and downloading the data, which is a little trickier on Colab than on your local machine using Jupyter. 

In [ ]:
# you will be prompted with a window asking to grant permissions
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# fill in the path in your Google Drive in the string below. Note: do not escape slashes or spaces
import os
datadir = "/content/assignment3"
if not os.path.exists(datadir):
  !ln -s "/content/drive/My Drive/path/to/your/assignment3" $datadir # TODO: Fill your assignment3 path
os.chdir(datadir)
!pwd

## Random State Setup

To ensure reproducible data loading and training, initialize a global random seed for all sources of randomness.

**Important:** Because notebooks allow cells to be executed in arbitrary order while sharing a single global random state, you should **call `seed_everything` immediately before running any long training process** to guarantee reproducible results.

In [ ]:
import random
import torch
import numpy as np


def seed_everything(seed: int = 0):
    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


SEED = 0
seed_everything(SEED)

## Device Setup

You are expected to run experiments for this assignment on GPU. Therefore, ensure GPU access is available and the code below shows `"cuda"` as the output.

**Important note:** If you are using Google Colab Pro, it is recommended to use the less expensive T4 GPU (the only option for Colab Free) for this assignment to avoid using up compute credits.

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

## Data Setup

The first thing to do is implement a dataset class to load rotated CIFAR10 images with matching labels. Since there is already a CIFAR10 dataset class implemented in `torchvision`, you will extend this class and modify the `__getitem__` method appropriately to load rotated images.

Below, you will implement a `rotate_image` function that rotates an image given a rotation label. Each rotation label should be an integer in the set {0, 1, 2, 3}, each of which corresponds to a rotation of 0, 90, 180, or 270 degrees, respectively.

Note the code setting up the train and test datasets with transformations. What type should the `image` argument in `rotate_image` be?

In [ ]:
from typing import Any, Callable, Literal
from pathlib import Path
from tqdm import tqdm

import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
import random


def rotate_image(image: Any, rotation_label: int):
    """Rotates an image. Each rotation label should be an integer in the set
    {0, 1, 2, 3}, each of which corresponds to a rotation of 0, 90, 180, or
    270 degrees, respectively.

    Args:
        image (Any): Input image.
        rotation_label (int): Label corresponding to rotation.
    """
    if rotation_label == 0:  # 0-degree rotation
        return image
    # TODO: Implement rotate_image()
    #
    #
    raise ValueError("Rotation should be 0, 90, 180, or 270 degrees")


class CIFAR10Rotation(CIFAR10):
    def __init__(
        self,
        root: str | Path,
        train: bool,
        download: bool,
        transform: Callable[[torch.Tensor], torch.Tensor],
    ):
        super().__init__(root=root, train=train, download=download, transform=transform)

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(
        self, index: int
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        image, class_label = super().__getitem__(index)

        # Randomly select image rotation
        rotation_label = random.choice([0, 1, 2, 3])
        image_rotated = rotate_image(image, rotation_label)

        rotation_label = torch.tensor(rotation_label).long()
        class_label = torch.tensor(class_label).long()
        return image, image_rotated, class_label, rotation_label

In [ ]:
transform_train = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ]
)

transform_test = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ]
)

batch_size = 128

train_dataset = CIFAR10Rotation(
    root="./data", train=True, download=True, transform=transform_train
)
train_dataloader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=0
)

test_dataset = CIFAR10Rotation(
    root="./data", train=False, download=True, transform=transform_test
)
test_dataloader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False, num_workers=0
)

The following code shows some example images and rotated images with labels from the dataloader:

In [ ]:
import matplotlib.pyplot as plt

classes = (
    "plane",
    "car",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
)

rotation_classes = ("0", "90", "180", "270")


def show_image(image: torch.Tensor, title: str = None):
    # Unnormalize CIFAR10-normalized tensor
    mean = torch.tensor([0.4914, 0.4822, 0.4465], device=image.device).view(3, 1, 1)
    std = torch.tensor([0.2023, 0.1994, 0.2010], device=image.device).view(3, 1, 1)
    image = image * std + mean
    image = image.clamp(0, 1).detach().cpu()

    np_image = image.numpy()
    plt.imshow(np.transpose(np_image, (1, 2, 0)))
    plt.axis("off")
    if title is not None:
        plt.title(title)
    plt.show()


data_iter = iter(train_dataloader)
images, rotated_images, class_labels, rotation_labels = next(data_iter)

# print images and rotated images
title = "Class labels: " + ", ".join([classes[class_labels[i]] for i in range(4)])
img_grid = show_image(torchvision.utils.make_grid(images[:4], padding=0), title=title)

title = "Rotation labels: " + ", ".join(
    [rotation_classes[rotation_labels[i]] for i in range(4)]
)
img_grid = show_image(
    torchvision.utils.make_grid(rotated_images[:4], padding=0), title=title
)

## Evaluation Code

For this assignment, you will write functions that can be reused for all experiments. Therefore, your evaluation and train functions will have to be able to handle both the self-supervised _rotation_ task and the fully supervised _classification_ task.

Now, you will implement a `run_test` function to compute classification accuracy and test loss.

In [ ]:
import torch.nn as nn


def run_test(
    model: nn.Module,
    test_dataloader: DataLoader,
    criterion: nn.Module,
    task: Literal["rotation", "classification"],
) -> tuple[float, float]:
    """
    Evaluate a model on a test dataset.

    Depending on the selected task, the function evaluates either
    rotation prediction (self-supervised task) or image classification.

    Args:
        model (nn.Module):
            The neural network to evaluate.

        test_dataloader (DataLoader):
            Dataloader yielding batches of
            `(images, rotated_images, class_labels, rotation_labels)`.

        criterion (nn.Module):
            Loss function used to compute the evaluation loss.

        task (Literal["rotation", "classification"]):
            Specifies which task to evaluate.
            - `"rotation"` uses rotated images and rotation labels.
            - `"classification"` uses original images and class labels.

    Returns:
        The test accuracy and average loss.
    """
    correct = 0
    total = 0
    avg_test_loss = 0.0

    with torch.no_grad():  # Disable gradient computation during evaluation
        for images, rotated_images, class_labels, rotation_labels in test_dataloader:
            if task == "rotation":
                images, labels = rotated_images.to(DEVICE), rotation_labels.to(DEVICE)
            elif task == "classification":
                images, labels = images.to(DEVICE), class_labels.to(DEVICE)
            # TODO: Calculate metrics by running images through the network.
            # The class with the highest energy is what we choose as prediction.
            #
            #
            #

            avg_test_loss += criterion(outputs, labels) / len(test_dataloader)

    test_acc = 100 * correct / total
    return test_acc.item(), avg_test_loss.item()

## 1. Train a ResNet18 on the self-supervised rotation task

In this section, you will train a ResNet18 model on the rotation task. The input is a rotated image, and the model predicts the rotation label.

In [ ]:
from torchvision.models import resnet18
import torch.optim as optim

model = resnet18(num_classes=4)
model = model.to(DEVICE)

An initial learning rate is provided for you to experiment with.

In [ ]:
learning_rate = 0.01
# TODO: Define criterion and optimizer
criterion = None
optimizer = None

# Schedule the initial LR to be decayed by 10x every `decay_epochs` epochs
decay_epochs = 15
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=decay_epochs, gamma=0.1)

Here, you will implement the main training loop, which will be used across all of your experiments.

In [ ]:
def train(
    model: nn.Module,
    train_dataloader: torch.utils.data.DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    num_epochs: int,
    task: Literal["rotation", "classification"],
    update_freq: int = 100,
    scheduler: optim.lr_scheduler._LRScheduler | None = None,
    test_dataloader: torch.utils.data.DataLoader | None = None,
) -> tuple[float, float]:
    """Train a model on either the rotation task or the classification task.

    Args:
        model (nn.Module): Model to train.
        train_dataloader (torch.utils.data.DataLoader): Dataloader for training.
        criterion (nn.Module): Loss.
        optimizer (optim.Optimizer): Optimizer.
        num_epochs (int): Number of training epochs.
        task (Literal["rotation", "classification"]): Which task to train the
            model on. You need to implement training for both tasks.
        update_freq (int, optional): How many steps it takes to update training
            metrics on the progress bar. Defaults to 100.
        scheduler (optim.lr_scheduler._LRScheduler | None, optional): Optional
            learning rate scheduler. Defaults to None.
        test_dataloader (torch.utils.data.DataLoader | None, optional): Dataloader
            for testing. Defaults to None (i.e., no testing).
    """
    epoch_pbar = tqdm(range(num_epochs), desc="Training")  # Training progress bar

    def update_pbar():
        epoch_pbar.set_postfix(
            epoch=epoch + 1,
            train_loss=f"{avg_loss:.3f}",
            train_acc=f"{avg_acc:.2f}%",
            test_loss=f"{test_loss:.3f}" if test_loss is not None else "N/A",
            test_acc=f"{test_acc:.2f}%" if test_acc is not None else "N/A",
        )

    update_freq = min(
        update_freq, len(train_dataloader)
    )  # update train metrics at least once per epoch

    avg_loss = avg_acc = None
    test_loss = test_acc = None

    for epoch in epoch_pbar:  # loop over the dataset multiple times
        running_loss = 0.0
        running_correct = 0.0
        running_total = 0.0

        model.train()

        for i, (images, rotated_images, class_labels, rotation_labels) in enumerate(
            train_dataloader
        ):
            # TODO: Set the data to the correct device; different tasks will use different inputs and labels

            # TODO: zero the parameter gradients

            # TODO: forward + backward + optimize

            # TODO: Get predicted results
            predicted = None

            # Print statistics
            running_loss += loss.item()
            running_total += labels.size(0)
            running_correct += (predicted == labels).sum().item()

            # Update progress bar every update_freq steps
            if (i + 1) % update_freq == 0:
                avg_loss = running_loss / update_freq
                avg_acc = 100.0 * running_correct / running_total

                update_pbar()

                running_loss = 0.0
                running_correct = 0
                running_total = 0

        # TODO: Run the run_test() function after each epoch; set the model to evaluation mode.
        #
        #
        
        if avg_loss is None:
            # update_freq is larger than dataloader length
            avg_loss = running_loss / len(train_dataloader)
            avg_acc = 100.0 * running_correct / running_total
        update_pbar()

        if scheduler is not None:
            scheduler.step()
    return test_acc, test_loss

Run the main training loop and save the model below. You need to save your models for all 5 parts since you will have to load them to obtain scores for the Kaggle competitions.

**Important note:** If you are using Colab Free, it is very important that you connect this runtime to your Google Drive using the cells at the beginning of the notebook, and save your model immediately after training to avoid losing your model in case the runtime/environment disconnects.

For reference, training this rotation prediction model using default hyperparameters on a T4 GPU (default GPU runtime on Colab Free) should take around 26 minutes. Aim for a baseline test accuracy of **78%**.

In [ ]:
seed_everything(SEED)
test_acc, _ = train(
    model,
    train_dataloader,
    criterion,
    optimizer,
    num_epochs=45,
    task="rotation",
    scheduler=scheduler,
    test_dataloader=test_dataloader,
)
print(f"Test accuracy: {test_acc:.3f}")

# TODO: Save the model

## 2. Finetuning the pretrained model on CIFAR10 classification

In this section, you will load the pretrained ResNet18 model and finetune it on the classification task. You will freeze all previous layers except for the `layer4` block and `fc` layer.

In [ ]:
# TODO: Load the pretrained ResNet18 model

In [ ]:
# TODO: Freeze all previous layers; only keep the 'layer4' block and 'fc' layer trainable

In [ ]:
# Print all the trainable parameters
print("Trainable parameters:")
params_to_update = []
for name, param in model.named_parameters():
    if param.requires_grad:
        params_to_update.append(param)
        print("-", name)

In [ ]:
learning_rate = 0.01
# TODO: Define criterion and optimizer
criterion = None
optimizer = None

# Schedule the initial LR to be decayed by 10x every `decay_epochs` epochs
decay_epochs = 10
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=decay_epochs, gamma=0.1)

Training this model over 20 epochs (default) on a Colab Free T4 GPU should take around 10 minutes. Aim for a baseline test accuracy of **60%**.

In [ ]:
seed_everything(SEED)
test_acc, _ = train(
    model,
    train_dataloader,
    criterion,
    optimizer,
    num_epochs=20,
    task="classification",
    scheduler=scheduler,
    test_dataloader=test_dataloader,
)
print(f"Test accuracy: {test_acc:.3f}")

# TODO: Save the model

## 3. Supervised training on a randomly initialized model
In this section, we will randomly initialize a ResNet18 model and train the whole model on the classification task.

In [ ]:
# TODO: Randomly initialize a new ResNet18 model

In [ ]:
learning_rate = 0.01
# TODO: Define criterion and optimizer
criterion = None
optimizer = None

# Schedule the initial LR to be decayed by 10 every `decay_epochs` epochs
decay_epochs = 10
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=decay_epochs, gamma=0.1)

Aim for a test accuracy baseline of **80%**.

In [ ]:
seed_everything(SEED)
test_acc, _ = train(
    model,
    train_dataloader,
    criterion,
    optimizer,
    num_epochs=20,
    task="classification",
    scheduler=scheduler,
    test_dataloader=test_dataloader,
)
print(f"Test accuracy: {test_acc:.3f}")

# TODO: Save the model

## 4. Supervised training on the pretrained model
In this section, you will load the pretrained ResNet18 model and re-train the whole model on the classification task.

In [ ]:
# TODO: Load the pretrained ResNet18 model

In [ ]:
learning_rate = 0.01
# TODO: Define criterion and optimizer
criterion = None
optimizer = None

# Schedule the initial LR to be decayed by 10x every `decay_epochs` epochs
decay_epochs = 10
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=decay_epochs, gamma=0.1)

Aim for a test accuracy baseline of **82%**.

In [ ]:
seed_everything(SEED)
test_acc, _ = train(
    model,
    train_dataloader,
    criterion,
    optimizer,
    num_epochs=20,
    task="classification",
    scheduler=scheduler,
    test_dataloader=test_dataloader,
)
print(f"Test accuracy: {test_acc:.3f}")

# TODO: Save the model

## 5. Experimenting with bells and whistles
In this section, you will experiment with an advanced architecture that is either off-the-shelf or custom-created, alongside any combination of "bells and whistles" (optimization, data augmentation, normalization, etc.) to get the highest score possible on the the fully supervised classification task. You should aim to have at least a **3%** increase from **best** model from parts 2, 3, and 4.

For this section, you can either (i) train your advanced model from scratch, or (ii) pretrain it on the rotation prediction task and finetune part or all of the pretrained model on CIFAR10 classification. However, you are **not allowed to initialize your model from off-the-shelf pretrained weights**.

In [ ]:
# TODO: Add bells and whistles
model = None

In [ ]:
# TODO: Define criterion and optimizer
criterion = None
optimizer = None

In [ ]:
# TODO: Start train loop and save model

## Extra Credit